In [1]:
# Required imports
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import Runnable
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory import ChatMessageHistory
from typing import Dict


In [3]:
# Setup model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Define prompt with placeholder for memory
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Create base runnable
chain: Runnable = prompt | model

# In-memory chat history store
store: Dict[str, ChatMessageHistory] = {}

# Memory-backed chain
memory_chain = RunnableWithMessageHistory(
    chain,
    lambda session_id: store.setdefault(session_id, ChatMessageHistory()),
    input_messages_key="input",
    history_messages_key="history"
)

# Define a prompt request structure for testing
class PromptRequest(BaseModel):
    prompt: str
    session_id: str

In [4]:
# Function to simulate interaction
def ask(prompt_text: str, session_id: str = "default"):
    request = PromptRequest(prompt=prompt_text, session_id=session_id)
    response = memory_chain.invoke(
        {"input": request.prompt},
        config={"configurable": {"session_id": request.session_id}}
    )
    return response.content

# Example usage
response = ask("What's the capital of France?")
print("Response:", response)

Response: The capital of France is Paris.
